In [2]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

C:\Users\ankit\AppData\Local\Temp\ipykernel_3860\1007429996.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


CUDA available: True


In [3]:
#importing dataset



# 1. Initialize the loader with your PDF file
loader = PyPDFLoader("S:/Projects/rag-dissertation/datasets/UNEP-Overshoot-Spotlight-Report.pdf")

# 2. Load and split the PDF into a list of Documents (one per page)
docs = loader.load()

# Or load and split into chunks directly
pages = loader.load_and_split()

print(f"Total pages loaded: {len(docs)}")
print(docs[0].page_content)  # Content of the first page
print(docs[0].metadata)  # Page number and source path

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/QUBIJR+ArialMT', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(1385, 0, 2954413007888), '/LastChar': 124, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [278, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 667, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 0, 0, 0, 556, 0, 556, 556, 222, 0, 0, 0, 0, 556, 556, 556, 0, 333, 500, 278, 0, 0, 0, 0, 500, 0, 0, 260]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/DDNIJR+RobotoCondensed-Bold', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(3600, 0, 2954413007888), '/LastChar': 114, '/Subtype': '/Type1', '/Type': '/Fon

Total pages loaded: 139
I    | UNEP |  Limiting Overshoot
Navigating exceedance of 1.5°C 
and pathways towards return
Limiting 
overshoot
{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.4 (Windows)', 'creationdate': '2026-09-02T06:48:49+03:00', 'author': 'UNEP', 'moddate': '2026-09-02T07:14:27+03:00', 'subject': 'Navigating exceedance of 1.5°C and pathways towards return.\r\n\r\nhttps://doi.org/10.59117/20.500.11822/49857', 'title': 'Limiting Overshoot', 'trapped': '/False', 'source': 'S:/Projects/rag-dissertation/datasets/UNEP-Overshoot-Spotlight-Report.pdf', 'total_pages': 139, 'page': 0, 'page_label': 'I'}


In [ ]:
#loading dataset in langchain

# docs = []
# for col in df.columns:
#     if col == "Unnamed: 0":
#         continue
#     text = "\n".join(df[col].dropna().astype(str))
#     docs.append(Document(page_content=text, metadata={"column": col}))

# print(len(docs))   # = number of columns
# print(docs[0].metadata)
# print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [4]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

606
preparation of this report. Authors and reviewers have 
contributed to the report in their individual capacities. 
Their affiliations are only mentioned for identification 
purposes.
Chapter Teams:
Executive Summary: Michael Logan (UNEP)
Introduction
Mirey Atallah (UNEP), David Dodman (IHS), Kristie L. Ebi 
(University of Washington, United States of America), 
Sharon Gil (UNEP), Elisabeth Gilmore (Carleton University, 
Canada), Paula A. Harrison (UK Centre for Ecology & 
Hydrology [UKCEH]), United Kingdom), Bruce Hewitson 
(UCT, South Africa), Zelina Zaiton Ibrahim (Universiti 
Putra Malaysia, Malaysia), Mariam Saleh Khan (Weather 
and Climate Services, Pakistan and Climate Analytics, 
Germany), Lena Kopetzky (UNEP), Nicolien de Lange 
(UNEP), Anita Lazurko (UKCEH, United Kingdom), Li-Jing 
Liu (Beijing Institute of Technology, China), Shobha 
Maharaj (University of Fiji, Fiji), Adrienne Marshall 
(Colorado School of Mines, United States of America),


In [5]:
#Embeddings Model 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 663.42it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [7]:
m = embeddings._client
print(m.max_seq_length)                  # 256
print(next(m.parameters()).dtype)        # torch.float16
print(next(m.parameters()).device)       # cuda:0

256
torch.float16
cuda:0


In [6]:
# Embeddings & vector storage
#
# This cell only READS the Chroma DB that was persisted to ./chroma_langchain_db
# (src/chroma_langchain_db) -- it does not re-embed the dataset on every run.

from pathlib import Path

CHROMA_DIR = Path("chroma_langchain_db")
if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"Local Chroma DB not found at {CHROMA_DIR.resolve()}. "
        "Run the ingestion cells once with REBUILD_INDEX = True to create it."
    )

vector_store = Chroma(
    collection_name="Science_Embeddings",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# Default LangChain retriever over the persisted collection.
retriever = vector_store.as_retriever()

print(f"Loaded {vector_store._collection.count()} chunks from {CHROMA_DIR.resolve()}")


Loaded 0 chunks from S:\Projects\rag-dissertation\src\chroma_langchain_db


In [7]:
# Timeline of storage
#
# The index already exists on disk, so ingestion is opt-in: flip REBUILD_INDEX to
# True only when you want to (re)embed the split texts into the collection.

from tqdm import tqdm

REBUILD_INDEX = True

if REBUILD_INDEX:
    max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
    BATCH = min(2000, max_bs)                            # stay safely under it

    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i + BATCH]
        ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
        vector_store.add_documents(batch, ids=ids)
else:
    print("Skipping ingestion; reading the existing local Chroma DB as-is.")


100%|██████████| 1/1 [01:36<00:00, 96.79s/it]


In [8]:
# Retrieve evidence for the same question that will be evaluated.
query = "What factors does cryosphere depends on which determins how they will response to 1.5 degree overshoot?"
retrieved_docs = retriever.invoke(query)
retrieval_context = [doc.page_content for doc in retrieved_docs]
if not retrieval_context:
    raise ValueError("The retriever returned no context; cannot evaluate a RAG answer.")
print(f"Retrieved {len(retrieval_context)} context chunks.")

# Show the retrieved chunks (rank, metadata, text) so the evidence is visible.
for rank, doc in enumerate(retrieved_docs, start=1):
    meta = ", ".join(f"{k}={v}" for k, v in doc.metadata.items()) or "none"
    print(f"--- chunk {rank}/{len(retrieved_docs)} [{meta}] ---")
    print(doc.page_content)
    print()


Retrieved 4 context chunks.
--- chunk 1/4 [source=S:/Projects/rag-dissertation/datasets/UNEP-Overshoot-Spotlight-Report.pdf, creator=Adobe InDesign 21.4 (Windows), title=Limiting Overshoot, creationdate=2026-09-02T06:48:49+03:00, page_label=34, moddate=2026-09-02T07:14:27+03:00, subject=Navigating exceedance of 1.5°C and pathways towards return.

https://doi.org/10.59117/20.500.11822/49857, page=46, total_pages=139, author=UNEP, producer=Adobe PDF Library 18.0, trapped=/False] ---
34    | UNEP |  Limiting Overshoot
sea-level rise—may also lose stability above 2°C of 
global warming (Winkelmann et al. 2026). Even if the ice 
sheets do not undergo a transition to a new state, they 
will likely contribute to several metres of sea-level rise, 
resulting in inundation of global coastal ecosystems and 
communities, depending on the warming and cooling 
rate, and final temperature reached (Bochow et al. 2023; 
Coulon et al. 2025; Stokes et al. 2025) after an initial 
overshoot.
Due to relativ

In [9]:
#Setting up LLM

llm = ChatOllama(
    model="qwen3.5:2b", 
    temperature = 0.5,
    reasoning=False,
    
)

In [10]:
#Pre Retrieval - Baseline
context_text = "\n\n".join(retrieval_context)

response = llm.invoke(query)
print(response.content)


The cryosphere (the frozen part of the Earth's climate system, including glaciers, ice sheets, sea ice, permafrost, and snowpack) is a highly sensitive component of the global climate system. Its response to a **1.5°C overshoot** (a scenario where global temperatures rise beyond the 1.5°C target, likely reaching 2°C or higher) depends on a complex interplay of physical, chemical, biological, and socio-economic factors.

Here are the key determinants that dictate how the cryosphere will respond:

### 1. Initial Temperature Rise Rate (Rate of Warming)
The speed at which temperatures rise is arguably the most critical factor. The cryosphere has a **time-dependent response**.
*   **Fast Warming:** If temperatures rise rapidly (e.g., >0.5°C per decade), ice sheets and glaciers can lose mass faster than they can accumulate due to melting, leading to rapid retreat.
*   **Slow Warming:** In contrast, if warming is gradual, the cryosphere may have time to adjust its geometry or thickness before

In [11]:
#Pre Retrieval - Baseline Eval

import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    """Ice sheets and other cryosphere ecosystems on the
frontline. These ecosystems are already experiencing
among the highest magnitudes and rates of climate
change impacts in the world. Different components of
the cryosphere (such as ice sheets (see above), sea ice,
glaciers, permafrost and snow) respond differently to
a temporary overshoot of 1.5°C, depending on climate
sensitivity, response behaviour, thresholds and lag times
of the cryospheric system elements, with irreversibility as
a potential outcome (Constable et al. 2022)."""
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="gemma4:e4b",
    base_url="http://localhost:11434",
    temperature=0,
    # reasoning=False,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\ankit\AppData\Local\Temp\ipykernel_3860\791538821.py:75: DeprecationWarning: 'HallucinationMetric' now scores in the same direction as every other deepeval metric: 1 is a pass, 0 is a failure, and 'threshold' is the MINIMUM passing score. It previously scored the proportion of violations, where 'threshold' was a maximum. Review any 'threshold' you pass and any code reading '.score' - a threshold of 0.2 that used to mean 'at most 20% violations' should now be 0.8. This notice will be removed in a future release.
  "hallucination": HallucinationMetric(


hallucination: 1.000
  passed: True
  reason: The score is 1.00 because the actual output accurately reflects the context's statement that different cryospheric components respond differently to a temporary overshoot of 1.5°C.


faithfulness: 1.000
  passed: True
  reason: The score is 1.00 because there are no contradictions, meaning the actual output is perfectly faithful to the retrieval context.


answer relevancy: 1.000
  passed: True
  reason: The score is 1.00 because the output directly addresses the factors determining the cryosphere's response to a 1.5°C overshoot, and there were no irrelevant statements found to detract from the score.


In [12]:
# Post Retrieval Query
context_text = "\n\n".join(retrieval_context)
prompt = f"""Answer the question using only the supplied context. If the context does not contain the answer, say so.

Context:
{context_text}

Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)


Based on the provided context, the **cryosphere** (specifically ice sheets and other cryosphere ecosystems) responds to a 1.5°C overshoot pathway based on the following factors:

*   **Final Temperature Reached:** The response depends on the final temperature reached after an initial overshoot, as this determines whether coastal ecosystems and communities will be inundated.
*   **Warming and Cooling Rates:** The magnitude of sea-level rise and subsequent ecosystem inundation is dependent on these rates.
*   **Regional Differences:** Impacts vary by region because overshoot affects temperatures and precipitation differently across the globe, with elevated surface temperatures particularly in northern latitudes during the peak stage.
*   **Persistence of Changes:** The response involves differences in how precipitation and temperature patterns persist after stabilization (e.g., whether they remain cooler in a decline stage compared to a monotonic warming scenario).


In [13]:
#Post Retrieval Eval
import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    """Ice sheets and other cryosphere ecosystems on the
    frontline. These ecosystems are already experiencing
    among the highest magnitudes and rates of climate
    change impacts in the world. Different components of
    the cryosphere (such as ice sheets (see above), sea ice,
    glaciers, permafrost and snow) respond differently to
    a temporary overshoot of 1.5°C, depending on climate
    sensitivity, response behaviour, thresholds and lag times
    of the cryospheric system elements, with irreversibility as
    a potential outcome (Constable et al. 2022)."""
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="gemma4:e4b",
    base_url="http://localhost:11434",
    temperature=0,
    # reasoning=False,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

hallucination: 0.000
  passed: False
  reason: The score is 0.00 because the actual output introduces several specific factors ('Final Temperature Reached', 'Warming and Cooling Rates', 'Regional Differences', and 'Persistence of Changes') that are not mentioned in the context, which only lists 'climate sensitivity, response behaviour, thresholds and lag times of the cryospheric system elements'.


faithfulness: 1.000
  passed: True
  reason: The score is 1.00 because there are no contradictions, meaning the actual output is perfectly faithful to the retrieval context.


answer relevancy: 1.000
  passed: True
  reason: The score is 1.00 because the output directly addresses the factors determining the cryosphere's response to a 1.5°C overshoot, and there were no irrelevant statements found to detract from the score.
